# Resaving parquet
The processed DWD weather data comes as zip files containing txt files. To process these efficiently, we first convert them the [parquet file format](https://en.wikipedia.org/wiki/Apache_Parquet).

In [1]:
import pandas as pd
import numpy as np
import stackview
from dwd_weather import load_station, load_index, load_timeseries, generate_plot_stack

In [2]:
rain_dataset = "rain_recent"
temp_dataset = "temp_recent"

In [3]:
stations_df = load_index(temp_dataset)
stations_df

,Stations_id,von_datum,bis_datum,Stationshoehe,geoBreite,geoLaenge,Stationsname,Bundesland,Abgabe
0,00044,20070208,20260704,44,52.9336,8.2370,Großenkneten,Niedersachsen,Frei
1,00073,20070214,20260704,374,48.6183,13.0620,Aldersbach-Kramersepp,Bayern,Frei
2,00078,20041011,20260703,64,52.4853,7.9125,Alfhausen,Niedersachsen,Frei
3,00091,20020820,20260704,304,50.7446,9.3450,Alsfeld-Eifa,Hessen,Frei
4,00096,20190409,20260704,50,52.9437,12.8518,Neuruppin-Alt Ruppin,Brandenburg,Frei
...,...,...,...,...,...,...,...,...,...
468,19171,20200820,20260704,13,54.0039,9.8555,Hasenkrug-Hardebek,Schleswig-Holstein,Frei
469,19172,20200820,20260704,48,54.0246,9.3880,Wacken,Schleswig-Holstein,Frei
470,19207,20230330,20260703,16,53.8178,12.0645,Gülzow-Prüzen,Mecklenburg-Vorpommern,Frei
471,19856,20240625,20260704,625,47.6134,12.9819,Schönau am Königssee,Bayern,Frei


In [4]:
station_ids = stations_df['Stations_id'].unique()
station_ids[:3]

<StringArray>
['00044', '00073', '00078']
Length: 3, dtype: str

In [5]:
temp_ts = load_timeseries(station_ids, temp_dataset, "TT_10")
temp_ts

100%|██████████| 473/473 [00:45<00:00, 10.43it/s]


,STATIONS_ID,MESS_DATUM,QN,PP_10,TT_10,TM5_10,RF_10,TD_10,eor,Stations_id
0,44,202412310000,3,-999.0,6.4,6.0,87.3,4.4,eor,00044
1,44,202412310010,3,-999.0,6.4,6.1,86.9,4.4,eor,00044
2,44,202412310020,3,-999.0,6.4,5.9,87.5,4.5,eor,00044
3,44,202412310030,3,-999.0,6.4,6.0,87.1,4.4,eor,00044
4,44,202412310040,3,-999.0,6.4,5.9,87.9,4.5,eor,00044
...,...,...,...,...,...,...,...,...,...,...
37074797,20098,202607032310,2,-999.0,15.3,9.8,44.3,3.2,eor,20098
37074798,20098,202607032320,2,-999.0,15.1,9.4,45.0,3.3,eor,20098
37074799,20098,202607032330,2,-999.0,15.0,7.3,45.6,3.4,eor,20098
37074800,20098,202607032340,2,-999.0,14.9,9.3,46.5,3.5,eor,20098


In [6]:
temp_ts.describe()

,STATIONS_ID,MESS_DATUM,QN,PP_10,TT_10,TM5_10,RF_10,TD_10
count,3.707480e+07,3.707480e+07,3.707480e+07,3.707480e+07,3.707480e+07,3.707480e+07,3.707480e+07,3.707480e+07
mean,4.020755e+03,2.025389e+11,2.998035e+00,-2.398731e+02,5.750848e+00,-1.356432e+01,7.203080e+01,3.119420e-01
std,3.242419e+03,4.616060e+07,4.428688e-02,9.623181e+02,6.188500e+01,1.506012e+02,7.079836e+01,6.832939e+01
min,4.400000e+01,2.024123e+11,1.000000e+00,-9.990000e+02,-9.990000e+02,-9.990000e+02,-9.990000e+02,-9.990000e+02
25%,1.721000e+03,2.025052e+11,3.000000e+00,-9.990000e+02,3.100000e+00,1.600000e+00,6.420000e+01,2.000000e-01
50%,3.490000e+03,2.025100e+11,3.000000e+00,-9.990000e+02,9.200000e+00,8.200000e+00,8.100000e+01,4.800000e+00
75%,5.433000e+03,2.026022e+11,3.000000e+00,9.741000e+02,1.510000e+01,1.520000e+01,9.160000e+01,9.900000e+00
max,2.009800e+04,2.026070e+11,3.000000e+00,1.044800e+03,4.140000e+01,5.190000e+01,1.000000e+02,2.780000e+01


In [7]:
merged_df = pd.merge(temp_ts, stations_df, on="Stations_id")
merged_df.head()

,STATIONS_ID,MESS_DATUM,QN,PP_10,TT_10,TM5_10,RF_10,TD_10,eor,Stations_id,von_datum,bis_datum,Stationshoehe,geoBreite,geoLaenge,Stationsname,Bundesland,Abgabe
0,44,202412310000,3,-999.0,6.4,6.0,87.3,4.4,eor,00044,20070208,20260704,44,52.9336,8.237,Großenkneten,Niedersachsen,Frei
1,44,202412310010,3,-999.0,6.4,6.1,86.9,4.4,eor,00044,20070208,20260704,44,52.9336,8.237,Großenkneten,Niedersachsen,Frei
2,44,202412310020,3,-999.0,6.4,5.9,87.5,4.5,eor,00044,20070208,20260704,44,52.9336,8.237,Großenkneten,Niedersachsen,Frei
3,44,202412310030,3,-999.0,6.4,6.0,87.1,4.4,eor,00044,20070208,20260704,44,52.9336,8.237,Großenkneten,Niedersachsen,Frei
4,44,202412310040,3,-999.0,6.4,5.9,87.9,4.5,eor,00044,20070208,20260704,44,52.9336,8.237,Großenkneten,Niedersachsen,Frei


In [11]:
merged_df['datetime'] = pd.to_datetime(merged_df['MESS_DATUM'].astype(int).astype(str), format='%Y%m%d%H%M')
merged_df.to_parquet(temp_dataset + ".parquet")

In [12]:
rain_ts = load_timeseries(station_ids, rain_dataset, "RWS_10")
merged_df2 = pd.merge(rain_ts, stations_df, on="Stations_id")
merged_df2['datetime'] = pd.to_datetime(merged_df2['MESS_DATUM'].astype(int).astype(str), format='%Y%m%d%H%M')
merged_df2.to_parquet(rain_dataset + ".parquet")

100%|██████████| 473/473 [00:37<00:00, 12.61it/s]


In [9]:


plot_stack = generate_plot_stack(temp_ts, stations_df, "Temperature", days_delta=14, min_value=-40, max_value=40)
print(f"Plot stack shape: {plot_stack.shape}")

stackview.slice(plot_stack)

100%|██████████| 2016/2016 [05:37<00:00,  5.97it/s]


Plot stack shape: (2016, 600, 500, 3)


## Plotting analogous rain data
Next, we redo the same using rain data.

In [4]:
# Display the resulting DataFrame
rain_ts = load_timeseries(station_ids, rain_dataset, "RWS_10")
rain_ts

100%|██████████| 1397/1397 [01:51<00:00, 12.49it/s]


,STATIONS_ID,MESS_DATUM,QN,RWS_DAU_10,RWS_10,RWS_IND_10,eor,Station_id,Wert
0,20,202412250000,3,-999,0.0,-999,eor,00020,0.0
1,20,202412250010,3,-999,0.0,-999,eor,00020,0.0
2,20,202412250020,3,-999,0.0,-999,eor,00020,0.0
3,20,202412250030,3,-999,0.0,-999,eor,00020,0.0
4,20,202412250040,3,-999,0.0,-999,eor,00020,0.0


In [6]:
rain_ts.describe()

,STATIONS_ID,MESS_DATUM,QN,RWS_DAU_10,RWS_10,RWS_IND_10,Wert
count,1.082902e+08,1.082902e+08,1.082902e+08,1.082902e+08,1.082902e+08,1.082902e+08,1.076438e+08
mean,6.202039e+03,2.025368e+11,2.997232e+00,-8.711775e+02,-5.951645e+00,-8.712927e+02,1.210043e-02
std,5.550657e+03,4.744807e+07,5.253618e-02,3.339001e+02,7.695648e+01,3.335978e+02,1.098347e-01
min,2.000000e+01,2.024122e+11,1.000000e+00,-9.990000e+02,-9.990000e+02,-9.990000e+02,0.000000e+00
25%,2.319000e+03,2.025051e+11,3.000000e+00,-9.990000e+02,0.000000e+00,-9.990000e+02,0.000000e+00
50%,4.490000e+03,2.025093e+11,3.000000e+00,-9.990000e+02,0.000000e+00,-9.990000e+02,0.000000e+00
75%,7.328000e+03,2.026021e+11,3.000000e+00,-9.990000e+02,0.000000e+00,-9.990000e+02,0.000000e+00
max,2.010700e+04,2.026063e+11,3.000000e+00,1.000000e+01,9.990000e+01,1.000000e+00,9.990000e+01


In [10]:
plot_stack = generate_plot_stack(rain_ts, df, "Rain", days_delta=14, min_value=0, max_value=5, colormap="Blues")
print(f"Plot stack shape: {plot_stack.shape}")

stackview.slice(plot_stack)

100%|██████████| 2016/2016 [07:55<00:00,  4.24it/s] 


Plot stack shape: (2016, 600, 500, 3)
